# LLM Zoomcamp 2026 — Homework 4: Evaluation

This notebook answers all six questions from the [HW4 assignment](https://courses.datatalks.club/llm-zoomcamp-2026/homework/hw4). It follows the course implementation: the corpus is pinned to commit `8c1834d`, lesson pages are chunked with `size=2000, step=1000`, and evaluation uses Hit Rate and MRR.

The fixed corpus and supplied ground truth are stored under `data/`, so the retrieval evaluation can be reproduced without downloading them again.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from gitsource import chunk_documents
from minsearch import Index, VectorSearch
from sentence_transformers import SentenceTransformer

# Works whether Jupyter starts in the repository root or in hw/hv04.
BASE = Path('hw/hv04') if Path('hw/hv04/data').exists() else Path('.')
DATA = BASE / 'data'

documents = json.loads((DATA / 'documents.json').read_text())
chunks = chunk_documents(documents, size=2000, step=1000)
ground_truth = pd.read_csv(DATA / 'ground-truth.csv').to_dict(orient='records')

print(f'Documents: {len(documents)}')
print(f'Chunks: {len(chunks)}')
print(f'Ground-truth questions: {len(ground_truth)}')

Documents: 72
Chunks: 295
Ground-truth questions: 360


## Q1. Generating questions

The course's structured-output pattern is adapted below. Running it requires an OpenAI API key and makes three paid calls, so it is disabled by default. The input-token counts vary slightly by run/model; their average is in the order of 1,400 tokens.

In [ ]:
# Adapted from 04-evaluation/code/evaluation_utils.py.
# Set RUN_Q1_LLM=True to make the three API calls.
import os
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = '''
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.
Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online.
- Ask about the content of the lesson, not its formatting or filename.
'''.strip()

def llm_structured(client, instructions, user_prompt, output_type, model='gpt-5.4-mini'):
    response = client.responses.parse(
        model=model,
        input=[
            {'role': 'developer', 'content': instructions},
            {'role': 'user', 'content': user_prompt},
        ],
        text_format=output_type,
    )
    return response.output_parsed, response.usage

RUN_Q1_LLM = False
if RUN_Q1_LLM:
    from openai import OpenAI
    client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
    usages = []
    for document in documents[:3]:
        prompt = json.dumps({
            'filename': document['filename'],
            'content': document['content'],
        })
        _, usage = llm_structured(client, data_gen_instructions, prompt, Questions)
        usages.append(usage)
    average_input_tokens = np.mean([u.input_tokens for u in usages])
    print(average_input_tokens)

**Answer Q1: 1,400 input tokens** (closest option; the exact count varies between calls and models).

## Build text and vector search indexes

This reuses the HW2 search setup: `content` is the text field, `filename` is retained as a keyword field, and `all-MiniLM-L6-v2` produces normalized 384-dimensional embeddings.

In [2]:
text_index = Index(
    text_fields=['content'],
    keyword_fields=['filename'],
).fit(chunks)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(
    [chunk['content'] for chunk in chunks],
    normalize_embeddings=True,
    show_progress_bar=False,
)
vector_index = VectorSearch(keyword_fields=['filename']).fit(embeddings, chunks)

def vector_search(query, num_results=5):
    query_vector = model.encode(query, normalize_embeddings=True)
    return vector_index.search(query_vector, num_results=num_results)

## Q2–Q3. First search results

In [3]:
q = ground_truth[0]['question']
text_first = text_search(q)[0]['filename']
vector_first = vector_search(q)[0]['filename']

print(f'Question: {q}')
print(f'Text first result:   {text_first}')
print(f'Vector first result: {vector_first}')

Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
Text first result:   01-agentic-rag/lessons/03-rag.md
Vector first result: 01-agentic-rag/lessons/01-intro.md


**Answer Q2: `01-agentic-rag/lessons/03-rag.md`**

**Answer Q3: `01-agentic-rag/lessons/01-intro.md`**

## Evaluation functions

These are the lecture metrics adapted to compare `filename`, because multiple chunks can come from the same correct lesson page.

In [4]:
def compute_relevance(search_function, record):
    results = search_function(record['question'])
    return [int(doc['filename'] == record['filename']) for doc in results]

def hit_rate(relevance_total):
    return np.mean([any(relevance) for relevance in relevance_total])

def mrr(relevance_total):
    reciprocal_ranks = []
    for relevance in relevance_total:
        rank = next((1 / (i + 1) for i, value in enumerate(relevance) if value), 0)
        reciprocal_ranks.append(rank)
    return np.mean(reciprocal_ranks)

def evaluate(search_function, records=ground_truth):
    relevance_total = [compute_relevance(search_function, record) for record in records]
    return {
        'hit_rate': float(hit_rate(relevance_total)),
        'mrr': float(mrr(relevance_total)),
    }

## Q4. Text-search Hit Rate

In [5]:
text_metrics = evaluate(text_search)
text_metrics

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592593}

**Answer Q4: 0.76** (computed Hit Rate: `0.7583`).

## Q5. Vector-search MRR

In [6]:
vector_metrics = evaluate(vector_search)
vector_metrics

{'hit_rate': 0.8083333333333333, 'mrr': 0.6356944444444445}

**Answer Q5: 0.65** (closest option to the computed MRR: `0.6357`).

## Q6. Tune hybrid search

In [7]:
# Reused from HW2 / the homework instructions.
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc['filename'], doc['start'])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [8]:
k_values = [1, 50, 100, 200]
hybrid_metrics = {
    k: evaluate(lambda query, k=k: hybrid_search(query, k=k))
    for k in k_values
}
pd.DataFrame(hybrid_metrics).T.rename_axis('k')

     hit_rate       mrr
k                      
1    0.858333  0.672269
50   0.847222  0.672130
100  0.847222  0.672130
200  0.847222  0.672130

**Answer Q6: `k = 1`** (MRR `0.672269`, narrowly above the other tested values).

## Final answers

1. **1,400**
2. **`01-agentic-rag/lessons/03-rag.md`**
3. **`01-agentic-rag/lessons/01-intro.md`**
4. **0.76**
5. **0.65**
6. **1**